# Dermato — YOLOv8 training on Colab GPU

Retrains `skin_problems` and `acne_detection` on a free T4 GPU instead of a CPU laptop (30h -> ~30-60min per run).

**Before running:** upload `face_skin_problems.zip` and `acne_detection.zip` to the root of your Google Drive
(My Drive / dermato_datasets/), or change `DRIVE_DATASET_DIR` below to wherever you put them.

**Runtime > Change runtime type > T4 GPU** must be selected before running.

## 1. Setup

In [ ]:
!pip install -q ultralytics
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

### Keep this session alive

Free-tier Colab disconnects a runtime after ~90 minutes without *browser* activity, even if a cell
is actively training — this is what cut off the last two runs at ~94 min and ~40 min, well before
either model finished or actually early-stopped. The cell below pings the page every 60s so the tab
never looks idle. **Run it once, then leave the tab open** (it can be in a background tab, just don't
close it) for the full training run below.

In [ ]:
%%javascript
// Dispatches a fake mousemove on the page every 60s so Colab's idle-disconnect
// timer never fires. Doesn't depend on any specific toolbar button, so it
// keeps working even if Colab's UI changes.
if (window._dermatoKeepAlive) {
  clearInterval(window._dermatoKeepAlive);
}
window._dermatoKeepAlive = setInterval(() => {
  document.body.dispatchEvent(new MouseEvent('mousemove'));
  console.log('Keep-alive ping: ' + new Date().toLocaleTimeString());
}, 60000);
console.log('Keep-alive started.');

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile

# Change this if you uploaded the zips somewhere else in your Drive
DRIVE_DATASET_DIR = '/content/drive/MyDrive/dermato_datasets'
OUTPUT_DIR = '/content/drive/MyDrive/dermato_results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('/content/dataset', exist_ok=True)

for zip_name in ['face_skin_problems.zip', 'acne_detection.zip']:
    src = os.path.join(DRIVE_DATASET_DIR, zip_name)
    assert os.path.exists(src), f'Missing {src} -- upload it to your Drive first'
    with zipfile.ZipFile(src) as z:
        z.extractall('/content/dataset')
    print('Extracted', zip_name)

!ls /content/dataset

In [ ]:
# Dataset YAMLs -- same class layout as the local training/configs/*.yaml, paths repointed at Colab

skin_problems_yaml = '''
path: /content/dataset/Face Skin Problems.v1i.yolo26
train: train/images
val: valid/images
test: test/images
nc: 10
names:
  0: Acne
  1: Blackheads
  2: Dark-Spots
  3: Dry-Skin
  4: Enlarged-Pores
  5: Eyebags
  6: Oily-Skin
  7: Skin-Redness
  8: Whiteheads
  9: Wrinkles
'''

acne_detection_yaml = '''
path: /content/dataset/Acne.v21i.yolo26
train: train/images
val: valid/images
test: test/images
nc: 6
names:
  0: blackheads
  1: dark spot
  2: nodules
  3: papules
  4: pustules
  5: whiteheads
'''

with open('/content/skin_problems.yaml', 'w') as f:
    f.write(skin_problems_yaml)
with open('/content/acne_detection.yaml', 'w') as f:
    f.write(acne_detection_yaml)

print('Wrote dataset YAMLs.')

## 2. Train skin_problems

Changes vs. the CPU run that scored mAP50 0.168:
- `yolov8s.pt` instead of `yolov8n.pt` (GPU affords the bigger backbone)
- `imgsz=960` instead of 640 -- Blackheads/Whiteheads are tiny, low-contrast lesions that need more pixels to survive downsampling
- `mixup=0.0` -- blending images muddies the subtle color/texture cues that separate Dry-Skin/Oily-Skin/Skin-Redness
- `patience=30` since GPU epochs are cheap, worth searching a bit longer

**Resumable:** this writes checkpoints straight to Google Drive (`OUTPUT_DIR`), which survives a
Colab disconnect even though `/content` itself is wiped. If this cell is interrupted, just reconnect,
re-run the setup/mount/dataset cells above, then re-run this cell — it detects the existing
`last.pt` in Drive and continues training from that epoch instead of starting over. To intentionally
start fresh with different hyperparameters, delete or rename `dermato_results/skin_problems` in
Drive first (`resume=True` reuses the hyperparameters saved with the checkpoint, it won't pick up
edits made below).

In [ ]:
import os
from ultralytics import YOLO

SKIN_RUN_DIR = os.path.join(OUTPUT_DIR, 'skin_problems')
SKIN_LAST_CKPT = os.path.join(SKIN_RUN_DIR, 'weights', 'last.pt')

if os.path.exists(SKIN_LAST_CKPT):
    print(f'Found existing checkpoint at {SKIN_LAST_CKPT} -- resuming training from there.')
    model = YOLO(SKIN_LAST_CKPT)
    results = model.train(resume=True)
else:
    print('No existing checkpoint found -- starting a fresh run.')
    model = YOLO('yolov8s.pt')
    results = model.train(
        data='/content/skin_problems.yaml',
        epochs=150,
        batch=16,
        imgsz=960,
        device=0,
        project=OUTPUT_DIR,
        name='skin_problems',
        exist_ok=True,

        hsv_h=0.02,
        hsv_s=0.6,
        hsv_v=0.4,
        fliplr=0.5,
        degrees=15.0,
        translate=0.1,
        scale=0.4,
        mosaic=1.0,
        mixup=0.0,

        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=3,
        patience=30,
        save_period=10,
        plots=True,
        verbose=True,
    )

print('mAP50:', results.results_dict.get('metrics/mAP50(B)'))
print('mAP50-95:', results.results_dict.get('metrics/mAP50-95(B)'))

## 3. Train acne_detection

Changes vs. the CPU run that scored mAP50 0.148:
- `yolov8s.pt`, `imgsz=960` for the same tiny-lesion reason (Blackheads AP50 was 0.024, Nodules 0.055)
- `flipud=0.0` -- vertical flips don't reflect how a face is ever photographed, likely just adding noise
- `patience=30`

**Resumable** the same way as the skin_problems cell above — safe to re-run after a disconnect,
it'll pick up from `dermato_results/acne_detection/weights/last.pt` in Drive if present.

In [ ]:
ACNE_RUN_DIR = os.path.join(OUTPUT_DIR, 'acne_detection')
ACNE_LAST_CKPT = os.path.join(ACNE_RUN_DIR, 'weights', 'last.pt')

if os.path.exists(ACNE_LAST_CKPT):
    print(f'Found existing checkpoint at {ACNE_LAST_CKPT} -- resuming training from there.')
    model2 = YOLO(ACNE_LAST_CKPT)
    results2 = model2.train(resume=True)
else:
    print('No existing checkpoint found -- starting a fresh run.')
    model2 = YOLO('yolov8s.pt')
    results2 = model2.train(
        data='/content/acne_detection.yaml',
        epochs=150,
        batch=16,
        imgsz=960,
        device=0,
        project=OUTPUT_DIR,
        name='acne_detection',
        exist_ok=True,

        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.3,
        fliplr=0.5,
        flipud=0.0,
        degrees=10.0,
        translate=0.1,
        scale=0.3,
        mosaic=1.0,

        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        patience=30,
        save_period=10,
        plots=True,
        verbose=True,
    )

print('mAP50:', results2.results_dict.get('metrics/mAP50(B)'))
print('mAP50-95:', results2.results_dict.get('metrics/mAP50-95(B)'))

## 4. Results

Weights + plots are saved directly to `dermato_results/skin_problems/weights/best.pt` and
`dermato_results/acne_detection/weights/best.pt` in your Google Drive -- download those two
`best.pt` files and drop them into the matching folders under `backend/models/.../weights/weights/`
on your laptop to replace the CPU-trained ones.

In [ ]:
print('skin_problems best.pt:', os.path.join(OUTPUT_DIR, 'skin_problems', 'weights', 'best.pt'))
print('acne_detection best.pt:', os.path.join(OUTPUT_DIR, 'acne_detection', 'weights', 'best.pt'))